<a href="https://colab.research.google.com/github/kartik2006-del/efficient-finetuning-lora/blob/main/lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LoRA Evaluation Script


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls -lah "/content/drive/MyDrive/efficient-finetuning-lora/checkpoint-2588"

total 24M
drwx------ 2 root root 4.0K Sep 26 11:09 .
drwx------ 3 root root 4.0K Sep 26 11:09 ..
-rw------- 1 root root 1.1K Sep 26 11:09 adapter_config.json
-rw------- 1 root root 4.2M Sep 26 11:09 adapter_model.safetensors
-rw------- 1 root root 2.5K Sep 26 11:09 chat_template.jinja
-rw------- 1 root root 8.4M Sep 26 11:09 optimizer.pt
-rw------- 1 root root 5.1K Sep 26 11:09 README.md
-rw------- 1 root root  15K Sep 26 11:09 rng_state.pth
-rw------- 1 root root 1.4K Sep 26 11:09 scaler.pt
-rw------- 1 root root 1.5K Sep 26 11:09 scheduler.pt
-rw------- 1 root root  691 Sep 26 11:09 tokenizer_config.json
-rw------- 1 root root  11M Sep 26 11:09 tokenizer.json
-rw------- 1 root root  48K Sep 26 11:09 trainer_state.json
-rw------- 1 root root 5.1K Sep 26 11:09 training_args.bin


In [4]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [5]:
import torch
import peft

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)

PyTorch: 2.11.0+cpu
PEFT: 0.20.0


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
from peft import PeftModel

adapter_path = "/content/drive/MyDrive/efficient-finetuning-lora/checkpoint-2588"

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feature

In [8]:
model.print_trainable_parameters()

trainable params: 0 || all params: 495,114,112 || trainable%: 0.0000


In [9]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned")

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [11]:
df = dataset["train"].to_pandas()

df = df.drop_duplicates(
    subset=["instruction", "input", "output"]
).reset_index(drop=True)

In [12]:
from datasets import Dataset

formatted_data = Dataset.from_pandas(df)

train_test = formatted_data.train_test_split(
    test_size=0.20,
    seed=42
)

train_data = train_test["train"]

temp_data = train_test["test"].train_test_split(
    test_size=0.50,
    seed=42
)

validation_data = temp_data["train"]
test_data = temp_data["test"]

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Train: 41404
Validation: 5176
Test: 5176


In [13]:
def generate_response(instruction, input_text=""):
    if input_text.strip():
        user_content = instruction + "\n\nInput:\n" + input_text
    else:
        user_content = instruction

    messages = [
        {"role": "user", "content": user_content}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [14]:
example = test_data[0]

print("INSTRUCTION:")
print(example["instruction"])

print("\nINPUT:")
print(example["input"])

print("\nEXPECTED OUTPUT:")
print(example["output"])

print("\nMODEL OUTPUT:")
print(
    generate_response(
        example["instruction"],
        example["input"]
    )
)

INSTRUCTION:
You are given a computer programming language, and you need to come up with an example program to demonstrate it.

INPUT:
Python

EXPECTED OUTPUT:
Here is an example program written in Python that will print the first 10 numbers in the Fibonacci sequence.

```python
# defining the function 
def fibonacci_seq(n):
    # assigning first two numbers of the sequence
    a = 0
    b = 1

    # check if the number of terms is zero
    if n <= 0:
        print("Invalid input. Enter a positive integer.")
    # check if the number of terms is one
    elif n == 1:
        print(a)
    else:
        #print the first two numbers
        print(a)
        print(b)
        # run a for loop to calculate the next numbers of the sequence
        for i in range(n-2):
            c = a + b
            a = b
            b = c
            print(c)

# Driver code to test the function
fibonacci_seq(10)
```

This program defines a function named `fibonacci_seq` which takes in one parameter `n` whic

In [16]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [17]:
import evaluate

rouge = evaluate.load("rouge")

In [18]:
import pandas as pd
import time

num_samples = 100

results = []

for i in range(num_samples):
    example = test_data[i]

    prediction = generate_response(
        example["instruction"],
        example["input"]
    )

    results.append({
        "instruction": example["instruction"],
        "input": example["input"],
        "reference": example["output"],
        "prediction": prediction
    })

    if (i + 1) % 10 == 0:
        print(f"Completed: {i + 1}/{num_samples}")

Completed: 10/100
Completed: 20/100
Completed: 30/100


KeyboardInterrupt: 

In [19]:
results_df = pd.DataFrame(results)

print(results_df.shape)
results_df.head()

(37, 4)


,instruction,input,reference,prediction
0,"You are given a computer programming language,...",Python,Here is an example program written in Python t...,Here is an example Python program that demonst...
1,"Given the parameters of a triangle, find out i...",Side 1 = 4\nSide 2 = 6\nSide 3 = 8,The perimeter of a triangle is the sum of the ...,The perimeter of a triangle is the sum of all ...
2,Describe a landscape in three phrases.,,1. Rolling hills dotted with wildflowers.\n2. ...,A beautiful mountain range with rolling hills ...
3,Summarize the difference between a tab and a s...,,A tab and a space are both white space charact...,A tab is a small horizontal line that is used ...
4,Write an original joke,,Why don't scientists trust atoms? Because they...,Why did the tomato turn red?\n\nBecause it saw...


In [20]:
scores = rouge.compute(
    predictions=results_df["prediction"].tolist(),
    references=results_df["reference"].tolist()
)

print(scores)

{'rouge1': np.float64(0.3195651058331882), 'rouge2': np.float64(0.10986980765930865), 'rougeL': np.float64(0.19749203812959198), 'rougeLsum': np.float64(0.2645655937231737)}


In [21]:
results_df["prediction_length"] = results_df["prediction"].apply(
    lambda x: len(x.split())
)

print(
    "Average generated length:",
    results_df["prediction_length"].mean()
)

Average generated length: 101.91891891891892


In [22]:
for i in range(5):
    print("=" * 80)
    print("INSTRUCTION:")
    print(results_df.iloc[i]["instruction"])

    print("\nREFERENCE:")
    print(results_df.iloc[i]["reference"])

    print("\nPREDICTION:")
    print(results_df.iloc[i]["prediction"])

INSTRUCTION:
You are given a computer programming language, and you need to come up with an example program to demonstrate it.

REFERENCE:
Here is an example program written in Python that will print the first 10 numbers in the Fibonacci sequence.

```python
# defining the function 
def fibonacci_seq(n):
    # assigning first two numbers of the sequence
    a = 0
    b = 1

    # check if the number of terms is zero
    if n <= 0:
        print("Invalid input. Enter a positive integer.")
    # check if the number of terms is one
    elif n == 1:
        print(a)
    else:
        #print the first two numbers
        print(a)
        print(b)
        # run a for loop to calculate the next numbers of the sequence
        for i in range(n-2):
            c = a + b
            a = b
            b = c
            print(c)

# Driver code to test the function
fibonacci_seq(10)
```

This program defines a function named `fibonacci_seq` which takes in one parameter `n` which represents the numb

In [23]:
import os

os.makedirs("results", exist_ok=True)

In [24]:
results_df.to_csv(
    "results/lora_evaluation_100.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [25]:
metrics = {
    "num_samples": 100,
    "rouge1": float(scores["rouge1"]),
    "rouge2": float(scores["rouge2"]),
    "rougeL": float(scores["rougeL"]),
    "rougeLsum": float(scores["rougeLsum"]),
    "avg_generated_length": float(
        results_df["prediction_length"].mean()
    )
}

metrics

{'num_samples': 100,
 'rouge1': 0.3195651058331882,
 'rouge2': 0.10986980765930865,
 'rougeL': 0.19749203812959198,
 'rougeLsum': 0.2645655937231737,
 'avg_generated_length': 101.91891891891892}

In [26]:
import json

with open("results/lora_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved!")

Metrics saved!


In [27]:
manual_analysis = pd.DataFrame(columns=[
    "example",
    "quality",
    "reason"
])

manual_analysis

,example,quality,reason


In [28]:
i = 0

print("INSTRUCTION:")
print(results_df.iloc[i]["instruction"])

print("\nREFERENCE:")
print(results_df.iloc[i]["reference"])

print("\nPREDICTION:")
print(results_df.iloc[i]["prediction"])

INSTRUCTION:
You are given a computer programming language, and you need to come up with an example program to demonstrate it.

REFERENCE:
Here is an example program written in Python that will print the first 10 numbers in the Fibonacci sequence.

```python
# defining the function 
def fibonacci_seq(n):
    # assigning first two numbers of the sequence
    a = 0
    b = 1

    # check if the number of terms is zero
    if n <= 0:
        print("Invalid input. Enter a positive integer.")
    # check if the number of terms is one
    elif n == 1:
        print(a)
    else:
        #print the first two numbers
        print(a)
        print(b)
        # run a for loop to calculate the next numbers of the sequence
        for i in range(n-2):
            c = a + b
            a = b
            b = c
            print(c)

# Driver code to test the function
fibonacci_seq(10)
```

This program defines a function named `fibonacci_seq` which takes in one parameter `n` which represents the numb

In [29]:
results_df.to_csv(
    "results/lora_evaluation_100.csv",
    index=False
)

print("Evaluation results saved!")

Evaluation results saved!


In [30]:
!ls -lh results/

total 60K
-rw-r--r-- 1 root root 54K Sep 27 18:10 lora_evaluation_100.csv
-rw-r--r-- 1 root root 215 Sep 27 18:09 lora_metrics.json
